# Impressão 3D na Engenharia Mecânica (Parte 1) — Análise Física e Classificação de Polímeros

## O Problema de Manufatura Aditiva (FDM)

Em processos de **Modelagem por Fusão e Deposição (FDM)**, os dois termoplásticos mais empregados na indústria de prototipagem são:
- **`ABS` (*Acrilonitrila Butadieno Estireno*):** Exige bico mais quente (`~220–250 °C`), mesa aquecida alta e **sofre contração térmica / delaminação (*warping*) quando resfriado rápido pela ventoinha**!
- **`PLA` (*Ácido Polilático*):** Funde em temperaturas menores (`~190–220 °C`) e **ganha excelente acabamento e resistência com ventilação ativa**.

O arquivo experimental `data.csv` reúne **50 corpos de prova ensaiados** contendo parâmetros de impressão e propriedades mecânicas resultantes:
- `layer_height` (mm), `wall_thickness` (mm), `infill_density` (%), `infill_pattern` (`grid` ou `honeycomb`), `nozzle_temperature` (°C), `bed_temperature` (°C), `print_speed` (mm/s), `fan_speed` (%), `roughness` (µm), `tension_strenght` (MPa), `elongation` (%) e `material` (`abs` ou `pla`).

Nesta **Parte 1**, vamos inspecionar fisicamente esses dados e treinar modelos (`KNN`, `MLP` do `Scikit-Learn` e `Keras` moderno) para identificar o polímero **sem vazamento de dados (*Data Leakage*)**!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

# Lê data.csv com o separador correto ',' (vírgula!)
df = pd.read_csv("data.csv", sep=",")

print(f"Dimensões do lote experimental: {df.shape[0]} corpos de prova x {df.shape[1]} colunas")
print(df.head())

---

## 1. Codificação de Variáveis Categóricas e Visualização Termomecânica

Convertemos `material` (`abs -> 0`, `pla -> 1`) e `infill_pattern` (`grid -> 0`, `honeycomb -> 1`) e plotamos dois gráficos fundamentais da Engenharia de Materiais:
1. **Velocidade da Ventoinha vs. Resistência à Tração** (mostrando como o vento degrada a adesão entre camadas do `ABS`!);
2. **Altura de Camada vs. Rugosidade Superficial** (efeito de "degrau de escada" na superfície da peça).

In [ ]:
# Codificação numérica limpa
df_num = df.copy()
df_num["material_cod"] = df_num["material"].map({"abs": 0, "pla": 1})
df_num["infill_pattern_cod"] = df_num["infill_pattern"].map({"grid": 0, "honeycomb": 1})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Subgráfico 1: Ventilação vs. Tração
sns.scatterplot(
    data=df, x="fan_speed", y="tension_strenght",
    hue="material", style="material", s=85, palette="Set1", ax=ax1
)
ax1.set_title("Ventilação vs. Resistência à Tração")
ax1.set_xlabel("Velocidade da Ventoinha (%)")
ax1.set_ylabel("Resistência à Tração (MPa)")

# Subgráfico 2: Altura de Camada vs. Rugosidade
sns.scatterplot(
    data=df, x="layer_height", y="roughness",
    hue="material", style="material", s=85, palette="Set1", ax=ax2
)
ax2.set_title("Altura de Camada vs. Rugosidade Superficial")
ax2.set_xlabel("Altura de Camada (mm)")
ax2.set_ylabel("Rugosidade (µm)")

plt.tight_layout()
plt.show()

**Leitura Física dos Gráficos:**
- No gráfico da esquerda, conforme `fan_speed` sobe de `0%` para `100%`, a resistência do **`ABS`** despenca porque o resfriamento brusco impede a difusão molecular entre filamentos adjacentes!
- No gráfico da direita, quanto maior `layer_height` (de `0.02 mm` até `0.20 mm`), maior a rugosidade superficial em **ambos** os materiais (relação quase linear).

---

## 2. Classificação Supervisionada (`ABS` vs. `PLA`) SEM Vazamento de Dados

> **Regra de Ouro:** Primeiro separamos `train_test_split` (`70%` treino, `30%` teste), e **só depois** ajustamos o `MinMaxScaler` no treino (`fit_transform` no treino, `transform` no teste)!

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

colunas_features = [
    "layer_height", "wall_thickness", "infill_density", "infill_pattern_cod",
    "nozzle_temperature", "bed_temperature", "print_speed", "fan_speed",
    "roughness", "tension_strenght", "elongation"
]

X = df_num[colunas_features].to_numpy()
y = df_num["material_cod"].to_numpy()  # 0 = ABS, 1 = PLA

# 1. Divisão estratificada ANTES do escalonamento!
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 2. Normalização Min-Max calibrada exclusivamente em X_train_raw
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# 3. Comparando KNN (k=5) vs. Rede Neural MLPClassifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
acc_knn = accuracy_score(y_test, knn.predict(X_test)) * 100.0

mlp = MLPClassifier(hidden_layer_sizes=(16, 8), activation="relu", solver="lbfgs", max_iter=1000, random_state=42)
mlp.fit(X_train, y_train)
acc_mlp = accuracy_score(y_test, mlp.predict(X_test)) * 100.0

print(f"Acurácia no Teste — KNN (k=5): {acc_knn:.2f}%")
print(f"Acurácia no Teste — Rede Neural MLP: {acc_mlp:.2f}%")

### E no `Keras` Moderno? (Corrigindo Erros Comuns em Classificação Binária)
Para uma decisão binária (`0 = ABS` ou `1 = PLA`), a camada final correta tem **1 único neurônio com ativação `sigmoid`** e função de perda `binary_crossentropy` — e para obter a classe prevista basta testar `(probabilidade >= 0.5).astype(int)`!

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

modelo_binario = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),  # 1 saída contínua = Probabilidade de ser PLA (1)
])

modelo_binario.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Passamos X_train JÁ NORMALIZADO (e nunca os dados brutos sem escala!)
modelo_binario.fit(X_train, y_train, epochs=150, batch_size=8, verbose=0)

probs_teste = modelo_binario(X_test, training=False).numpy().ravel()
classes_previstas = (probs_teste >= 0.5).astype(int)
acc_k = accuracy_score(y_test, classes_previstas) * 100.0
print(f"Acurácia no Teste — Keras Binário Moderno: {acc_k:.2f}%")

---

## 3. Exercícios Práticos de Fixação

### Exercício 1: Qual Parâmetro Sozinho Já Diferencia `ABS` de `PLA`?
Plote um `sns.boxplot(data=df, x='material', y='nozzle_temperature', hue='material', legend=False)` e observe as faixas de temperatura de bico usadas para extrudar cada material!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
plt.figure(figsize=(6, 4.5))
sns.boxplot(data=df, x="material", y="nozzle_temperature", hue="material", palette="Set2", legend=False)
plt.title("Temperatura do Bico Extrusor por Material")
plt.xlabel("Polímero")
plt.ylabel("Temperatura do Bico (°C)")
plt.tight_layout()
plt.show()
# Nota de Engenharia: Como o ABS exige temperaturas >= 220 °C na maioria dos ensaios e o PLA <= 220 °C,
# 'nozzle_temperature' explica por que a rede neural atinge até 100% de acurácia!
```

</details>

### Exercício 2: Classificando Sem a "Cola" da Temperatura do Bico!
E se você receber uma peça **já pronta na bancada** (sem saber em qual temperatura ela foi impressa), medindo apenas `roughness`, `tension_strenght` e `elongation`? Treine um `MLPClassifier` usando apenas essas 3 propriedades mecânicas finais e meça a acurácia no teste!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
X_bancada = df_num[["roughness", "tension_strenght", "elongation"]].to_numpy()
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_bancada, y, test_size=0.30, random_state=42, stratify=y)

sc_b = MinMaxScaler()
mlp_b = MLPClassifier(hidden_layer_sizes=(12, 6), solver="lbfgs", max_iter=1000, random_state=42)
mlp_b.fit(sc_b.fit_transform(X_tr_b), y_tr_b)

acc_bancada = mlp_b.score(sc_b.transform(X_te_b), y_te_b) * 100.0
print(f"Acurácia identificando o polímero apenas por ensaio mecânico: {acc_bancada:.2f}%")
```

</details>

---

## Próximo Passo
Na indústria, mais importante ainda do que descobrir o material é **prever quantos MPa a peça aguentará antes mesmo de ligar a impressora** (Gêmeo Digital). Vamos construir esse regressor na **[Parte 2 — Predição de Resistência Mecânica](02_impressora3d_predicao_resistencia_mecanica.ipynb)**!